# Blacknaught: training on IBM LinuxONE
Fits the outage-duration model and the district fault-risk baseline **on this z17 VM**, from 312,936 real Electricity North West outages (CC BY 4.0) downloaded straight onto LinuxONE.

In [1]:
import platform, subprocess, sys
print('machine:', platform.machine(), '| byte order:', sys.byteorder)
print(subprocess.run("lscpu | grep -E 'Machine type|Model name'", shell=True, capture_output=True, text=True).stdout)
print('NNPA:', 'yes' if 'nnpa' in open('/proc/cpuinfo').read() else 'no')

machine: s390x | byte order: big
Model name:                              -
Machine type:                            9175

NNPA: yes


In [2]:
import sys, time
sys.path.insert(0, '/home/ibm-user/shared/blacknaught/backend')
from pathlib import Path
from app import survival, arearisk
DATA = Path('/home/ibm-user/shared/blacknaught/data/enwl_unplanned-outages.csv')
print(DATA.name, round(DATA.stat().st_size / 1e6, 1), 'MB')

enwl_unplanned-outages.csv 83.8 MB


## 1. Outage-duration model (Kaplan–Meier fallback until M2)

In [3]:
t0 = time.perf_counter()
km_doc = survival.fit_from_enwl(DATA)
print(f"fitted on {km_doc['n']:,} outages in {time.perf_counter() - t0:.1f} s on {platform.machine()}")
model = survival.KmModel(km_doc['curve'], km_doc['model_id'], 'notebook')
for elapsed in (0, 1, 4, 12):
    p = model.predict(elapsed)
    print(f'outage already {elapsed:>2} h old → remaining q50 {p["quantiles"]["q50"]:6.2f} h, '
          f'q90 {p["quantiles"]["q90"]:7.2f} h, P(still off in 4 h) {p["surv"][3]:.2f}')

fitted on 312,936 outages in 1.9 s on s390x
outage already  0 h old → remaining q50   3.38 h, q90   15.90 h, P(still off in 4 h) 0.45
outage already  1 h old → remaining q50   3.54 h, q90   16.69 h, P(still off in 4 h) 0.46
outage already  4 h old → remaining q50   4.71 h, q90   33.46 h, P(still off in 4 h) 0.55
outage already 12 h old → remaining q50   7.99 h, q90  163.47 h, P(still off in 4 h) 0.69


## 2. District fault-risk baseline (climatology until M8)

In [4]:
t0 = time.perf_counter()
clim = arearisk.fit(DATA)
print(f"{len(clim['rates'])} districts, {clim['years']}, fitted in {time.perf_counter() - t0:.1f} s")
from datetime import date
for r in arearisk.area_risk(clim, date(2026, 10, 19))[:6]:
    print(f"{r['district']:<12} expected {r['expected']:.2f} faults/day   P(≥3) {r['p_ge_k']:.2f}")

12 districts, [1997, 2026], fitted in 2.0 s
Ashton       expected 3.23 faults/day   P(≥3) 0.63
Barrow       expected 0.00 faults/day   P(≥3) 0.00
Blackburn    expected 3.21 faults/day   P(≥3) 0.62
Bolton       expected 4.24 faults/day   P(≥3) 0.80
Carlisle     expected 3.16 faults/day   P(≥3) 0.61
Kendal       expected 3.31 faults/day   P(≥3) 0.64


## 3. Two houses, one real fault
Same outage, different answers: who runs out of backup first.

In [5]:
from app import harm
pred = model.predict(1.0)
for name, reserve in (('House A: oxygen, cylinder 3 h left', 3.0), ('House B: ventilator, 14 h battery', 14.0)):
    a = harm.assess(pred['quantiles'], pred['surv'], reserve, 5, False)
    print(f"{name:<38} → {'ESCALATE' if a.escalate else 'SMS check'} | {'; '.join(a.why)}")

House A: oxygen, cylinder 3 h left     → ESCALATE | backup, reserve not ample → q75; q75 8.3 h > reserve 3.0 h
House B: ventilator, 14 h battery      → SMS check | reserve 14.0 h > 2 × q50 → q50; q50 3.5 h ≤ reserve 14.0 h
